# HW2 - Multiple Object Video Detection (VisDrone)

In [ ]:
SLUG = "multiple-object-video-detection-hw-2-ucu-ai-2026"
COMP = "/content/comp"
WORK = "/content/drive/MyDrive/hw2"
DATA = "/content/data"

SEED = 42
TRAIN_STEP = 20
VAL_STEP = 5
MAX_SIDE = 1600
IMGSZ = 1280

DEBUG = False

EPOCHS_YOLO = 1 if DEBUG else 12
EPOCHS_TV = 1 if DEBUG else 4
EPOCHS_DETR = 1 if DEBUG else 6

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi -L
!pip install -q -U ultralytics pycocotools kaggle

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, glob
from getpass import getpass

if not glob.glob(f"{COMP}/**/test_images.csv", recursive=True):
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ").strip()
    !kaggle competitions download -c {SLUG} -p /content/raw
    !unzip -q -o /content/raw/{SLUG}.zip -d {COMP}
    !rm -f /content/raw/{SLUG}.zip

found = glob.glob(f"{COMP}/**/test_images.csv", recursive=True)
assert found, "Download failed: check the token and that you joined the competition (see the kaggle message above)"
COMP = os.path.dirname(found[0])
print(COMP, os.listdir(COMP))

In [ ]:
import os, io, json, time, math, random, contextlib, logging, shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image, ImageReadMode
from torchvision.ops import box_iou, batched_nms, generalized_box_iou_loss
from ultralytics import YOLO, RTDETR

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_all()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
for d in ["runs", "figs", "subs"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. EDA

In [ ]:
def load_coco(split):
    with open(f"{COMP}/{split}/annotations_coco.json") as f:
        coco = json.load(f)
    imgs = pd.DataFrame(coco["images"])
    anns = pd.DataFrame(coco["annotations"])
    anns[["x", "y", "w", "h"]] = pd.DataFrame(anns["bbox"].tolist(), index=anns.index)
    anns = anns.merge(imgs[["id", "video", "width", "height", "frame_id"]].rename(columns={"id": "image_id"}),
                      on="image_id")
    imgs["split"] = split
    anns["split"] = split
    return coco, imgs, anns

coco_tr, imgs_tr, anns_tr = load_coco("train")
coco_va, imgs_va, anns_va = load_coco("val")
imgs = pd.concat([imgs_tr, imgs_va], ignore_index=True)
anns = pd.concat([anns_tr, anns_va], ignore_index=True)
anns["category"] = anns.category_id.map({c["id"]: c["name"] for c in coco_tr["categories"]})
test_df = pd.read_csv(f"{COMP}/test_images.csv")

print("train:", len(imgs_tr), "frames,", imgs_tr.video.nunique(), "videos")
print("val:  ", len(imgs_va), "frames,", imgs_va.video.nunique(), "videos")
print("test: ", len(test_df), "frames,", test_df["clip"].nunique(), "clips")
print("videos both in train and val:", set(imgs_tr.video) & set(imgs_va.video))

### Classes and ignored regions

In [ ]:
print(anns.groupby(["orig_category_id", "iscrowd"]).category.agg(["size", "nunique"]))

real = anns[anns.iscrowd == 0].copy()
crowd = anns[(anns.iscrowd == 1) & (anns.category_id == 1)]
print("\nreal objects:", real.groupby("split").size().to_dict())
print("ignore regions:", crowd.groupby("split").size().to_dict())
print("frames with ignore regions:",
      (crowd.groupby("split").image_id.nunique() / imgs.groupby("split").size()).round(2).to_dict())

In [ ]:
cnt = real.groupby(["category", "split"]).size().unstack().sort_values("train", ascending=False)
share = (cnt / cnt.sum()).round(3)
share.plot.bar(figsize=(8, 4), rot=30, title="class share")
plt.tight_layout(); plt.savefig(f"{WORK}/figs/classes.png", dpi=110); plt.show()
pd.concat([cnt, share], axis=1, keys=["count", "share"])

### Resolution and frames

In [ ]:
res = pd.concat([imgs[["split", "width", "height"]], test_df[["width", "height"]].assign(split="test")])
print(res.groupby(["width", "height", "split"]).size().unstack(fill_value=0)[["train", "val", "test"]])

step = imgs.sort_values(["video", "frame_id"]).groupby("video").frame_id.diff().dropna()
print("\nframe_id step:", step.value_counts().to_dict())
print(imgs.groupby(["split", "video"]).size().groupby("split").describe()[["mean", "min", "max"]])
print("val frames per video:", imgs_va.groupby("video").size().sort_values(ascending=False).to_dict())

### Object sizes

In [ ]:
real["size"] = np.sqrt(real.w * real.h)
real["size_cat"] = pd.cut(real.w * real.h, [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])

plt.figure(figsize=(8, 4))
for split in ["train", "val"]:
    plt.hist(real[real.split == split]["size"], bins=np.logspace(0, 3, 60), alpha=0.6, density=True, label=split)
plt.axvline(32, c="k", ls="--"); plt.axvline(96, c="k", ls="--")
plt.xscale("log"); plt.xlabel("sqrt(w*h), px"); plt.legend(); plt.title("object size")
plt.tight_layout(); plt.savefig(f"{WORK}/figs/sizes.png", dpi=110); plt.show()

print(pd.crosstab(real.category, real.size_cat, normalize="index").round(2))
print(real.size_cat.value_counts(normalize=True).round(3).to_dict())

for s in [640, 960, 1280, 1600]:
    px = real["size"] * s / real[["width", "height"]].max(axis=1)
    print(f"imgsz {s}: <8px {(px < 8).mean():.1%}, <16px {(px < 16).mean():.1%}, median {px.median():.1f}px")

### Objects per frame

In [ ]:
n_obj = real.groupby(["split", "image_id"]).size()
print(n_obj.groupby("split").describe()[["mean", "50%", "max"]])

per_video = real.groupby(["split", "video", "image_id"]).size().groupby(["split", "video"]).mean().sort_values()
plt.figure(figsize=(8, 6))
plt.barh(range(len(per_video)), per_video.values,
         color=["tab:orange" if s == "val" else "tab:blue" for s in per_video.index.get_level_values(0)])
plt.yticks([]); plt.xlabel("mean objects per frame"); plt.title("density per video (orange = val)")
plt.tight_layout(); plt.savefig(f"{WORK}/figs/density.png", dpi=110); plt.show()

### GT boxes (and COCO vs YOLO label check)

In [ ]:
def draw_gt(split, video, frame, ax):
    im = imgs[(imgs.split == split) & (imgs.video == video) & (imgs.frame_id == frame)].iloc[0]
    ax.imshow(Image.open(f"{COMP}/{split}/{im.file_name}"))
    obj = real[(real.split == split) & (real.image_id == im.id)]
    ign = crowd[(crowd.split == split) & (crowd.image_id == im.id)]
    for r in obj.itertuples():
        ax.add_patch(patches.Rectangle((r.x, r.y), r.w, r.h, fill=False, lw=0.8, ec=plt.cm.tab10(r.category_id - 1)))
    for r in ign.itertuples():
        ax.add_patch(patches.Rectangle((r.x, r.y), r.w, r.h, color="red", alpha=0.35))
    n_yolo = sum(1 for l in open(f"{COMP}/{split}/labels/{video}/{frame:07d}.txt") if l.strip())
    ax.set_title(f"{video} {im.width}x{im.height}: coco={len(obj)} yolo={n_yolo} ignore={len(ign)}", fontsize=9)
    ax.axis("off")

dense = per_video.index[-1][1]
sparse = per_video.index[0][1]
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for ax, (s, v) in zip(axes.ravel(), [("train", dense), ("train", sparse),
                                      ("train", imgs_tr.video.iloc[0]), ("val", imgs_va.video.iloc[0])]):
    draw_gt(s, v, int(imgs[(imgs.split == s) & (imgs.video == v)].frame_id.min()), ax)
plt.tight_layout(); plt.savefig(f"{WORK}/figs/gt.png", dpi=90); plt.show()

In [ ]:
del anns, anns_tr, anns_va, real
import gc; gc.collect()

## 2. Data preparation

In [ ]:
def prepare(out, mask_ignore=False):
    if os.path.exists(f"{out}/data.yaml"):
        return out
    for split, step, coco in [("train", TRAIN_STEP, coco_tr), ("val", VAL_STEP, coco_va)]:
        images = [dict(im) for im in coco["images"] if (im["frame_id"] - 1) % step == 0]
        keep = {im["id"] for im in images}
        ann = [a for a in coco["annotations"] if a["image_id"] in keep]
        ign = defaultdict(list)
        for a in ann:
            if a["iscrowd"] == 1 and a["category_id"] == 1:
                ign[a["image_id"]].append(a["bbox"])

        def one(im):
            s = min(1.0, MAX_SIDE / max(im["width"], im["height"]))
            im["scale"] = s
            img = Image.open(f"{COMP}/{split}/{im['file_name']}").convert("RGB")
            if s < 1:
                img = img.resize((round(im["width"] * s), round(im["height"] * s)), Image.BILINEAR)
            if mask_ignore and split == "train":
                d = ImageDraw.Draw(img)
                for x, y, w, h in ign[im["id"]]:
                    d.rectangle([x * s, y * s, (x + w) * s, (y + h) * s], fill=(114, 114, 114))
            dst = f"{out}/{split}/{im['file_name']}"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            img.save(dst, quality=95)
            lab_src = f"{COMP}/{split}/{im['file_name']}".replace("/images/", "/labels/").replace(".jpg", ".txt")
            lab_dst = dst.replace("/images/", "/labels/").replace(".jpg", ".txt")
            os.makedirs(os.path.dirname(lab_dst), exist_ok=True)
            if os.path.exists(lab_src):
                shutil.copy(lab_src, lab_dst)
            else:
                open(lab_dst, "w").close()

        with ThreadPoolExecutor(8) as ex:
            list(ex.map(one, images))
        with open(f"{out}/{split}_gt.json", "w") as f:
            json.dump({"images": images, "annotations": ann, "categories": coco["categories"]}, f)
        print(split, len(images), "frames")
    with open(f"{out}/data.yaml", "w") as f:
        f.write(f"path: {out}\ntrain: train/images\nval: val/images\nnames: {CLASSES}\n")
    return out

t = time.time()
D0 = prepare(f"{DATA}/base")
GT = f"{D0}/val_gt.json"
print(f"{time.time() - t:.0f}s")

In [ ]:
def val_items(d):
    gt = json.load(open(f"{d}/val_gt.json"))
    return [(im["id"], f"{d}/val/{im['file_name']}", im["scale"]) for im in gt["images"]]

VAL = val_items(D0)
TEST = [(r.image_id, f"{COMP}/{r.file_name}", 1.0) for r in test_df.itertuples()]
if DEBUG:
    VAL, TEST = VAL[:30], TEST[:30]

_gt = json.load(open(GT))
GT_OBJ, GT_IGN = defaultdict(list), defaultdict(list)
for a in _gt["annotations"]:
    if a["iscrowd"] == 0:
        GT_OBJ[a["image_id"]].append(a["bbox"] + [a["category_id"]])
    elif a["category_id"] == 1:
        GT_IGN[a["image_id"]].append(a["bbox"])
VAL_IMG = {im["id"]: im for im in _gt["images"]}
print(len(VAL), "val frames,", len(TEST), "test frames")

## 3. Evaluation, submission, helpers

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate(preds, gt_json=GT):
    if len(preds) == 0:
        return dict(mAP=0, AP50=0, AP75=0, AP_small=0, AP_medium=0, AP_large=0)
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json)
        E = COCOeval(gt, gt.loadRes(preds), "bbox")
        E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    r = dict(mAP=s[0], AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4], AP_large=s[5])
    prec = E.eval["precision"]
    for k, c in enumerate(E.params.catIds):
        p = prec[:, :, k, 0, -1]
        r["AP_" + CLASSES[c - 1]] = p[p > -1].mean() if (p > -1).any() else float("nan")
    return {k: round(float(v), 4) for k, v in r.items()}

RESULTS = pd.read_csv(f"{WORK}/results.csv", index_col=0).to_dict("index") if os.path.exists(f"{WORK}/results.csv") else {}
def log(name, family, metrics, **extra):
    RESULTS[name] = dict(family=family, **metrics, **extra)
    pd.DataFrame(RESULTS).T.to_csv(f"{WORK}/results.csv")
    return pd.DataFrame([RESULTS[name]], index=[name])

def speed(predict_one, paths, n=50):
    for p in paths[:5]:
        predict_one(p)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t = time.time()
    for p in paths[5:5 + n]:
        predict_one(p)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.time() - t) / len(paths[5:5 + n]) * 1000
    return dict(ms=round(ms, 1), FPS=round(1000 / ms, 1))

def write_sub(preds, path):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    rows = []
    for iid in test_df.image_id:
        ps = sorted(by[iid], key=lambda p: -p["score"])[:500]
        rows.append(" ".join(f"{p['category_id'] - 1} {p['score']:.4f} {p['bbox'][0]:.1f} {p['bbox'][1]:.1f} "
                             f"{p['bbox'][2]:.1f} {p['bbox'][3]:.1f}" for p in ps))
    pd.DataFrame({"image_id": test_df.image_id, "PredictionString": rows}).to_csv(path, index=False)
    print("saved", path)

COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms", "FPS"]

In [ ]:
def iou_xywh(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = torch.tensor(np.c_[a[:, :2], a[:, :2] + a[:, 2:]])
    b = torch.tensor(np.c_[b[:, :2], b[:, :2] + b[:, 2:]])
    return box_iou(a, b).numpy()

def match_frame(iid, preds, thr=0.3):
    g = np.array(GT_OBJ[iid], float).reshape(-1, 5)
    ign = np.array(GT_IGN[iid], float).reshape(-1, 4)
    d = sorted([p for p in preds if p["score"] >= thr], key=lambda p: -p["score"])
    found = np.zeros(len(g), bool)
    types = []
    for p in d:
        iou = iou_xywh([p["bbox"]], g[:, :4])[0] if len(g) else np.zeros(0)
        same = g[:, 4] == p["category_id"] if len(g) else np.zeros(0, bool)
        cand = np.where(same & ~found & (iou >= 0.5))[0]
        if len(cand):
            found[cand[iou[cand].argmax()]] = True
            types.append("TP")
        elif len(ign) and iou_xywh([p["bbox"]], ign).max() > 0:
            x, y, w, h = p["bbox"]
            cx, cy = x + w / 2, y + h / 2
            inside = ((ign[:, 0] <= cx) & (cx <= ign[:, 0] + ign[:, 2]) & (ign[:, 1] <= cy) & (cy <= ign[:, 1] + ign[:, 3])).any()
            types.append("ignored" if inside else "background")
        elif len(g) and (iou[same] >= 0.5).any():
            types.append("duplicate")
        elif len(g) and (iou >= 0.5).any():
            types.append("wrong class")
        elif len(g) and (iou[same] >= 0.1).any():
            types.append("bad box")
        else:
            types.append("background")
    return d, types, g, found

def error_analysis(preds, thr=0.3):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    types, miss = defaultdict(int), defaultdict(lambda: [0, 0])
    worst = []
    for iid in VAL_IMG:
        d, t, g, found = match_frame(iid, by[iid], thr)
        for x in t:
            types[x] += 1
        for (x, y, w, h, c), f in zip(g, found):
            s = "small" if w * h < 32**2 else "medium" if w * h < 96**2 else "large"
            miss[s][0 if f else 1] += 1
        worst.append(((~found).sum() + sum(x not in ("TP", "ignored") for x in t), iid))
    print("detections:", dict(types))
    print("GT found / missed:", {k: f"{v[0]}/{v[1]} ({v[1] / max(sum(v), 1):.0%} missed)" for k, v in miss.items()})
    return [iid for _, iid in sorted(worst, reverse=True)]

def show_errors(preds, ids, title, thr=0.3):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    fig, axes = plt.subplots(len(ids) // 2, 2, figsize=(20, 6 * (len(ids) // 2)))
    for ax, iid in zip(np.ravel(axes), ids):
        im = VAL_IMG[iid]; s = im["scale"]
        ax.imshow(Image.open(f"{D0}/val/{im['file_name']}"))
        d, t, g, found = match_frame(iid, by[iid], thr)
        for x, y, w, h in GT_IGN[iid]:
            ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, color="grey", alpha=0.4))
        for (x, y, w, h, c), f in zip(g, found):
            if not f:
                ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, fill=False, ec="yellow", lw=1.2))
        for p, tt in zip(d, t):
            if tt != "ignored":
                x, y, w, h = p["bbox"]
                ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, fill=False, lw=0.8,
                                               ec="lime" if tt == "TP" else "red"))
        ax.set_title(f"{im['video']} #{im['frame_id']}  TP={t.count('TP')} FP={sum(x not in ('TP', 'ignored') for x in t)} "
                     f"missed={(~found).sum()}", fontsize=10)
        ax.axis("off")
    fig.suptitle(title + "  (green TP, red FP, yellow missed, grey ignore)")
    plt.tight_layout(); plt.savefig(f"{WORK}/figs/{title}.png", dpi=80); plt.show()

def size_plot(df, title):
    df[["AP_small", "AP_medium", "AP_large", "mAP"]].astype(float).plot.bar(figsize=(9, 4), rot=0, title=title)
    plt.tight_layout(); plt.savefig(f"{WORK}/figs/{title}.png", dpi=110); plt.show()

## 4. Torchvision models (Faster R-CNN, RetinaNet): data and training loop

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead, RetinaNetHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection import roi_heads

class TVData(Dataset):
    def __init__(self, gt_json, root, train=True):
        gt = json.load(open(gt_json))
        self.images, self.root, self.train = gt["images"], root, train
        self.boxes = defaultdict(list)
        for a in gt["annotations"]:
            x, y, w, h = a["bbox"]
            if a["iscrowd"] == 0 and w > 1 and h > 1:
                self.boxes[a["image_id"]].append([x, y, x + w, y + h, a["category_id"]])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        im = self.images[i]
        img = read_image(f"{self.root}/{im['file_name']}", ImageReadMode.RGB)
        b = torch.tensor(self.boxes[im["id"]], dtype=torch.float32).reshape(-1, 5)
        boxes, labels = b[:, :4] * im["scale"], b[:, 4].long()
        if self.train and random.random() < 0.5:
            img = img.flip(-1)
            W = img.shape[-1]
            boxes = torch.stack([W - boxes[:, 2], boxes[:, 1], W - boxes[:, 0], boxes[:, 3]], 1)
        return img, {"boxes": boxes, "labels": labels}

def collate(batch):
    return tuple(zip(*batch))

def tv_predict(model, items, bs=4):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(items), bs):
            chunk = items[i:i + bs]
            imgs = [read_image(p, ImageReadMode.RGB).to(DEVICE).float() / 255 for _, p, _ in chunk]
            with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                outs = model(imgs)
            for (key, _, s), o in zip(chunk, outs):
                for (x1, y1, x2, y2), sc, c in zip((o["boxes"].float() / s).tolist(), o["scores"].tolist(), o["labels"].tolist()):
                    if c > 0:
                        preds.append(dict(image_id=key, category_id=int(c), score=round(sc, 5),
                                          bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)]))
    return preds

def tv_speed(model):
    model.eval()
    def one(p):
        with torch.no_grad(), torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            model([read_image(p, ImageReadMode.RGB).to(DEVICE).float() / 255])
    return speed(one, [p for _, p, _ in VAL])

def tv_train(model, name, epochs=EPOCHS_TV, lr=0.005, bs=4):
    seed_all()
    model.to(DEVICE)
    ckpt, hist_csv = f"{WORK}/runs/{name}.pt", f"{WORK}/runs/{name}_hist.csv"
    if os.path.exists(ckpt) and os.path.exists(hist_csv):
        print(name, "already trained - loading")
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        return model, pd.read_csv(hist_csv)
    loader = DataLoader(TVData(f"{D0}/train_gt.json", f"{D0}/train"), batch_size=bs, shuffle=True,
                        num_workers=4, collate_fn=collate, drop_last=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    n_it = len(loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1, (it + 1) / 300) * (0.1 if it >= (epochs - 1) * n_it and epochs > 1 else 1))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    hist, best = [], -1
    for ep in range(epochs):
        model.train()
        t, tot = time.time(), 0
        for i, (imgs, tg) in enumerate(loader):
            if DEBUG and i == 5:
                break
            imgs = [x.to(DEVICE).float() / 255 for x in imgs]
            tg = [{k: v.to(DEVICE) for k, v in d.items()} for d in tg]
            with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                loss = sum(model(imgs, tg).values())
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 10)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        m = evaluate(tv_predict(model, VAL))
        hist.append(dict(epoch=ep + 1, loss=tot / (i + 1), time_h=(time.time() - t) / 3600, **m))
        print(f"{name} ep {ep + 1}: loss {tot / (i + 1):.3f}  mAP {m['mAP']:.4f}  AP50 {m['AP50']:.4f}  {(time.time() - t) / 60:.0f} min")
        if m["mAP"] > best:
            best = m["mAP"]
            torch.save(model.state_dict(), f"{WORK}/runs/{name}.pt")
    model.load_state_dict(torch.load(f"{WORK}/runs/{name}.pt"))
    pd.DataFrame(hist).to_csv(hist_csv, index=False)
    return model, pd.DataFrame(hist)

## 5. Two-stage: Faster R-CNN (ResNet-50 FPN, COCO weights)

In [ ]:
def frcnn(**kw):
    m = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=IMGSZ, max_size=IMGSZ,
                                box_detections_per_img=500, box_score_thresh=0.01, **kw)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 9)
    return m

ORIG_LOSS = roi_heads.fastrcnn_loss

def decode(d, w=(10., 10., 5., 5.)):
    dx, dy = d[:, 0] / w[0], d[:, 1] / w[1]
    dw, dh = (d[:, 2] / w[2]).clamp(max=math.log(1000 / 16)), (d[:, 3] / w[3]).clamp(max=math.log(1000 / 16))
    pw, ph = dw.exp(), dh.exp()
    return torch.stack([dx - pw / 2, dy - ph / 2, dx + pw / 2, dy + ph / 2], 1)

def giou_loss(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels)
    regression_targets = torch.cat(regression_targets)
    cls_loss = F.cross_entropy(class_logits, labels)
    pos = torch.where(labels > 0)[0]
    box_regression = box_regression.reshape(class_logits.shape[0], -1, 4)
    pred = decode(box_regression[pos, labels[pos]].float())
    target = decode(regression_targets[pos].float())
    box_loss = 10 * generalized_box_iou_loss(pred, target, reduction="sum") / labels.numel()
    return cls_loss, box_loss

FR_CFG = {
    "frcnn_base": {},
    "frcnn_dense": dict(rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=3000,
                        rpn_pre_nms_top_n_test=2000, rpn_post_nms_top_n_test=2000, box_positive_fraction=0.5),
}
fr_hist = {}

def run_frcnn(name, cfg, giou=False):
    roi_heads.fastrcnn_loss = giou_loss if giou else ORIG_LOSS
    model, fr_hist[name] = tv_train(frcnn(**cfg), name, lr=0.005)
    roi_heads.fastrcnn_loss = ORIG_LOSS
    preds = tv_predict(model, VAL)
    return model, preds, log(name, "faster_rcnn", {**evaluate(preds), **tv_speed(model)}, giou=giou, cfg=str(cfg))

In [ ]:
model, preds_fr_base, row = run_frcnn("frcnn_base", FR_CFG["frcnn_base"])
row[COLS]

In [ ]:
model, preds_fr_dense, row = run_frcnn("frcnn_dense", FR_CFG["frcnn_dense"])
row[COLS]

In [ ]:
fr_best_cfg = "frcnn_dense" if RESULTS["frcnn_dense"]["mAP"] > RESULTS["frcnn_base"]["mAP"] else "frcnn_base"
print("giou on top of", fr_best_cfg)
model, preds_fr_giou, row = run_frcnn("frcnn_giou", FR_CFG[fr_best_cfg], giou=True)
row[COLS]

In [ ]:
fr = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] == "faster_rcnn"}).T
display(fr[COLS])
display(fr[["AP_" + c for c in CLASSES]].astype(float).round(3))
size_plot(fr, "faster_rcnn_by_size")
for k, h in fr_hist.items():
    plt.plot(h.epoch, h.mAP, marker="o", label=k)
plt.legend(); plt.xlabel("epoch"); plt.ylabel("val mAP"); plt.show()

### RPN vs second stage

In [ ]:
fr_best = fr.mAP.astype(float).idxmax()
cfg = FR_CFG.get(fr_best, FR_CFG[fr_best_cfg])
model = frcnn(**cfg).to(DEVICE)
model.load_state_dict(torch.load(f"{WORK}/runs/{fr_best}.pt"))
model.eval()

model.rpn._pre_nms_top_n["testing"] = 2000
model.rpn._post_nms_top_n["testing"] = 2000
hit, total = defaultdict(int), 0
with torch.no_grad():
    for key, path, s in VAL[:200]:
        g = np.array(GT_OBJ[key], float).reshape(-1, 5)
        if len(g) == 0:
            continue
        img = read_image(path, ImageReadMode.RGB).to(DEVICE).float() / 255
        images, _ = model.transform([img])
        props, _ = model.rpn(images, model.backbone(images.tensors))
        h, w = images.image_sizes[0]
        p = props[0].cpu().numpy() * np.array([img.shape[2] / w, img.shape[1] / h] * 2) / s
        p = np.c_[p[:, :2], p[:, 2:] - p[:, :2]]
        for n in [100, 300, 1000, 2000]:
            iou = iou_xywh(g[:, :4], p[:n])
            hit[n] += int((iou.max(1) >= 0.5).sum()) if iou.shape[1] else 0
        total += len(g)
print("RPN recall@IoU0.5:", {n: round(v / total, 3) for n, v in hit.items()})

rows = {}
for n in [300, 1000, 2000]:
    model.rpn._pre_nms_top_n["testing"] = max(n, 1000)
    model.rpn._post_nms_top_n["testing"] = n
    rows[n] = {**evaluate(tv_predict(model, VAL)), **tv_speed(model)}
pd.DataFrame(rows).T[["mAP", "AP50", "AP_small", "ms"]]

### Failure cases

In [ ]:
model.rpn._pre_nms_top_n["testing"] = cfg.get("rpn_pre_nms_top_n_test", 1000)
model.rpn._post_nms_top_n["testing"] = cfg.get("rpn_post_nms_top_n_test", 1000)
preds = tv_predict(model, VAL)
worst = error_analysis(preds)
show_errors(preds, worst[:4], "frcnn_failures")
write_sub(tv_predict(model, TEST), f"{WORK}/subs/{fr_best}.csv")
del model; torch.cuda.empty_cache()

## 6. One-stage anchor-based: RetinaNet (ResNet-50 FPN, COCO weights)

In [ ]:
tr_gt = json.load(open(f"{D0}/train_gt.json"))
tr_im = {im["id"]: im for im in tr_gt["images"]}
wh = np.array([(a["bbox"][2] * IMGSZ / max(tr_im[a["image_id"]]["width"], tr_im[a["image_id"]]["height"]),
                a["bbox"][3] * IMGSZ / max(tr_im[a["image_id"]]["width"], tr_im[a["image_id"]]["height"]))
               for a in tr_gt["annotations"] if a["iscrowd"] == 0 and a["bbox"][2] > 1 and a["bbox"][3] > 1])
sz = np.sqrt(wh[:, 0] * wh[:, 1])
print("size percentiles 5/25/50/75/95:", np.percentile(sz, [5, 25, 50, 75, 95]).round(1))

base = int(np.clip(2 ** round(math.log2(np.percentile(sz, 5))), 8, 32))
DATA_SIZES = tuple(tuple(int(base * 2**l * 2**(k / 3)) for k in range(3)) for l in range(5))

lr_ = np.log(wh[:, 1] / wh[:, 0])
c = np.quantile(lr_, [0.2, 0.5, 0.8])
for _ in range(30):
    a = np.abs(lr_[:, None] - c[None]).argmin(1)
    c = np.array([lr_[a == k].mean() for k in range(3)])
DATA_RATIOS = (tuple(sorted(np.exp(c).round(2))),) * 5

DEF_SIZES = tuple((x, int(x * 2**(1 / 3)), int(x * 2**(2 / 3))) for x in [32, 64, 128, 256, 512])
DEF_RATIOS = ((0.5, 1.0, 2.0),) * 5
print("default:", DEF_SIZES, DEF_RATIOS[0])
print("data:   ", DATA_SIZES, DATA_RATIOS[0])

def coverage(sizes, ratios):
    A = np.array([(s / np.sqrt(r), s * np.sqrt(r)) for lv, rs in zip(sizes, ratios) for s in lv for r in rs])
    inter = np.minimum(wh[:, None, 0], A[None, :, 0]) * np.minimum(wh[:, None, 1], A[None, :, 1])
    best = (inter / (wh[:, 0:1] * wh[:, 1:2] + A[None, :, 0] * A[None, :, 1] - inter)).max(1)
    return {"IoU>=0.5": round((best >= 0.5).mean(), 3), "IoU>=0.4": round((best >= 0.4).mean(), 3),
            "small IoU>=0.5": round((best[sz < 32] >= 0.5).mean(), 3)}

print("coverage default:", coverage(DEF_SIZES, DEF_RATIOS))
print("coverage data:   ", coverage(DATA_SIZES, DATA_RATIOS))
print("anchors per 1280x720 frame:", sum(math.ceil(720 / s) * math.ceil(1280 / s) for s in [8, 16, 32, 64, 128]) * 9)

In [ ]:
def retina(anchors=None, reg_loss="l1"):
    m = retinanet_resnet50_fpn(weights="DEFAULT", min_size=IMGSZ, max_size=IMGSZ,
                               detections_per_img=500, score_thresh=0.01)
    if anchors:
        m.anchor_generator = AnchorGenerator(*anchors)
        m.head = RetinaNetHead(256, m.anchor_generator.num_anchors_per_location()[0], 9)
    else:
        m.head.classification_head = RetinaNetClassificationHead(256, m.head.classification_head.num_anchors, 9)
    m.head.regression_head._loss_type = reg_loss
    return m

RT_CFG = {"retina_base": None, "retina_anchors": (DATA_SIZES, DATA_RATIOS)}
rt_hist = {}

def run_retina(name, anchors, reg_loss="l1"):
    model, rt_hist[name] = tv_train(retina(anchors, reg_loss), name, lr=0.0025)
    preds = tv_predict(model, VAL)
    return model, preds, log(name, "retinanet", {**evaluate(preds), **tv_speed(model)},
                             anchors="data" if anchors else "default", reg_loss=reg_loss)

In [ ]:
model, preds_rt_base, row = run_retina("retina_base", None)
row[COLS]

In [ ]:
model, preds_rt_anch, row = run_retina("retina_anchors", RT_CFG["retina_anchors"])
row[COLS]

In [ ]:
rt_best_cfg = "retina_anchors" if RESULTS["retina_anchors"]["mAP"] > RESULTS["retina_base"]["mAP"] else "retina_base"
print("giou on top of", rt_best_cfg)
model, preds_rt_giou, row = run_retina("retina_giou", RT_CFG[rt_best_cfg], reg_loss="giou")
row[COLS]

In [ ]:
rt = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] in ("retinanet", "faster_rcnn")}).T
display(rt[COLS])
display(rt[["AP_" + c for c in CLASSES]].astype(float).round(3))
size_plot(rt, "retinanet_vs_frcnn_by_size")
for k, h in rt_hist.items():
    plt.plot(h.epoch, h.mAP, marker="o", label=k)
plt.legend(); plt.xlabel("epoch"); plt.ylabel("val mAP"); plt.show()

In [ ]:
rt_best = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] == "retinanet"}).T.mAP.astype(float).idxmax()
model = retina(RT_CFG.get(rt_best, RT_CFG[rt_best_cfg])).to(DEVICE)
model.load_state_dict(torch.load(f"{WORK}/runs/{rt_best}.pt"))
preds = tv_predict(model, VAL)
worst = error_analysis(preds)
show_errors(preds, worst[:2], "retinanet_errors")
write_sub(tv_predict(model, TEST), f"{WORK}/subs/{rt_best}.csv")
del model; torch.cuda.empty_cache()

## 7. Ultralytics helpers (YOLO, RT-DETR)

In [ ]:
def ul_train(name, src, imgsz, epochs, batch, load=None, cls=YOLO, data=None, **kw):
    run = f"{WORK}/runs/{name}"
    best, csv = f"{run}/weights/best.pt", f"{run}/results.csv"
    if os.path.exists(best) and os.path.exists(csv) and len(pd.read_csv(csv)) >= epochs:
        print(name, "already trained - loading")
        return cls(best)
    model = cls(src)
    if load:
        model.load(load)
    logging.getLogger("ultralytics").setLevel(logging.WARNING)
    t = time.time()
    model.train(data=data or f"{D0}/data.yaml", imgsz=imgsz, epochs=epochs, batch=batch, seed=SEED,
                project="/content/runs", name=name, exist_ok=True, workers=2, plots=False,
                fraction=0.1 if DEBUG else 1.0, **kw)
    print(f"{name}: {(time.time() - t) / 60:.0f} min")
    tr = model.trainer
    ckpt = str(tr.best if tr.best.exists() else tr.last)
    shutil.copytree(tr.save_dir, run, dirs_exist_ok=True)
    shutil.copy(ckpt, best)
    print("saved:", best, os.path.getsize(best) // 2**20, "MB")
    return cls(ckpt)

def ul_predict(model, items, imgsz, iou=0.6, augment=False, bs=8):
    preds = []
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        res = model.predict([p for _, p, _ in chunk], imgsz=imgsz, conf=0.001, iou=iou, max_det=500,
                            augment=augment, verbose=False)
        for (key, _, s), r in zip(chunk, res):
            for (x1, y1, x2, y2), sc, c in zip((r.boxes.xyxy.cpu().numpy() / s).tolist(),
                                               r.boxes.conf.tolist(), r.boxes.cls.tolist()):
                preds.append(dict(image_id=key, category_id=int(c) + 1, score=round(sc, 5),
                                  bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)]))
    return preds

def ul_speed(model, imgsz):
    return speed(lambda p: model.predict(p, imgsz=imgsz, conf=0.001, max_det=500, verbose=False),
                 [p for _, p, _ in VAL])

def ul_curve(name):
    df = pd.read_csv(f"{WORK}/runs/{name}/results.csv")
    df.columns = [c.strip() for c in df.columns]
    return df

## 8. One-stage anchor-free: YOLOv8

In [ ]:
YOLO_RUNS = {
    "yolo_base": dict(src="yolov8s.pt", imgsz=1280, batch=8),
    "yolo_1600": dict(src="yolov8s.pt", imgsz=1600, batch=4),
    "yolo_p2":   dict(src="yolov8s-p2.yaml", load="yolov8s.pt", imgsz=1280, batch=4),
}
yolo_preds = {}

def run_yolo(name):
    c = YOLO_RUNS[name]
    m = ul_train(name, c["src"], c["imgsz"], EPOCHS_YOLO, c["batch"], load=c.get("load"),
                 optimizer="SGD", lr0=0.01, close_mosaic=3, max_det=500, cache=False)
    yolo_preds[name] = ul_predict(m, VAL, c["imgsz"])
    return log(name, "yolo", {**evaluate(yolo_preds[name]), **ul_speed(m, c["imgsz"])}, imgsz=c["imgsz"], src=c["src"])

In [ ]:
run_yolo("yolo_base")[COLS]

In [ ]:
write_sub(ul_predict(YOLO(f"{WORK}/runs/yolo_base/weights/best.pt"), TEST, 1280), f"{WORK}/submission.csv")

In [ ]:
run_yolo("yolo_1600")[COLS]

In [ ]:
run_yolo("yolo_p2")[COLS]

In [ ]:
yo = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] == "yolo"}).T
display(yo[COLS])
display(yo[["AP_" + c for c in CLASSES]].astype(float).round(3))
size_plot(yo, "yolo_by_size")

allm = pd.DataFrame(RESULTS).T
display(allm[["family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms"]])

for k in YOLO_RUNS:
    df = ul_curve(k)
    plt.plot(df.epoch, df["metrics/mAP50-95(B)"], marker="o", label=k)
plt.legend(); plt.xlabel("epoch"); plt.ylabel("val mAP50-95 (ultralytics, no ignore regions)"); plt.show()

In [ ]:
yo_best = yo.mAP.astype(float).idxmax()
print("best:", yo_best)
worst = error_analysis(yolo_preds[yo_best])
show_errors(yolo_preds[yo_best], worst[:2], "yolo_errors")
m = YOLO(f"{WORK}/runs/{yo_best}/weights/best.pt")
write_sub(ul_predict(m, TEST, YOLO_RUNS[yo_best]["imgsz"]), f"{WORK}/subs/{yo_best}.csv")

## 9. Query-based: RT-DETR

In [ ]:
import yaml, ultralytics
cfg = yaml.safe_load(open(os.path.join(os.path.dirname(ultralytics.__file__), "cfg/models/rt-detr/rtdetr-l.yaml")))
cfg["head"][-1][3] = ["nc", 256, 500]
yaml.safe_dump(cfg, open(f"{WORK}/rtdetr-l-q500.yaml", "w"), sort_keys=False)

DETR_RUNS = {
    "rtdetr_base":   dict(src="rtdetr-l.pt"),
    "rtdetr_q500":   dict(src=f"{WORK}/rtdetr-l-q500.yaml", load="rtdetr-l.pt"),
    "rtdetr_frozen": dict(src="rtdetr-l.pt", freeze=10),
}
detr_preds = {}

def run_detr(name):
    c = dict(DETR_RUNS[name])
    src, load = c.pop("src"), c.pop("load", None)
    m = ul_train(name, src, IMGSZ, EPOCHS_DETR, 4, load=load, cls=RTDETR, optimizer="AdamW", lr0=1e-4,
                 weight_decay=1e-4, warmup_epochs=1, warmup_bias_lr=0.0, close_mosaic=2, cache=False, **c)
    detr_preds[name] = ul_predict(m, VAL, IMGSZ, bs=4)
    return log(name, "rtdetr", {**evaluate(detr_preds[name]), **ul_speed(m, IMGSZ)})

In [ ]:
run_detr("rtdetr_base")[COLS]

In [ ]:
run_detr("rtdetr_q500")[COLS]

In [ ]:
run_detr("rtdetr_frozen")[COLS]

In [ ]:
de = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] == "rtdetr"}).T
display(de[COLS])
display(de[["AP_" + c for c in CLASSES]].astype(float).round(3))
size_plot(de, "rtdetr_by_size")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k in list(DETR_RUNS) + ["yolo_base"]:
    df = ul_curve(k)
    ax[0].plot(df.epoch, df["metrics/mAP50-95(B)"], marker="o", label=k)
    ax[1].plot(df["time"] / 3600 if "time" in df else df.epoch, df["metrics/mAP50-95(B)"], marker="o", label=k)
ax[0].set_xlabel("epoch"); ax[1].set_xlabel("hours"); ax[0].legend()
plt.savefig(f"{WORK}/figs/rtdetr_vs_yolo.png", dpi=110); plt.show()

In [ ]:
n_gt = pd.Series({k: len(v) for k, v in GT_OBJ.items()})
plt.hist(n_gt, bins=40, alpha=0.6, label="GT objects")
for k, p in detr_preds.items():
    plt.hist(pd.Series([x["image_id"] for x in p if x["score"] > 0.3]).value_counts(), bins=40, alpha=0.4, label=k)
plt.axvline(300, c="k", ls="--"); plt.axvline(500, c="k", ls="--"); plt.legend(); plt.show()
print("max GT per val frame:", n_gt.max())

de_best = de.mAP.astype(float).idxmax()
worst = error_analysis(detr_preds[de_best])
show_errors(detr_preds[de_best], worst[:2], "rtdetr_errors")
write_sub(ul_predict(RTDETR(f"{WORK}/runs/{de_best}/weights/best.pt"), TEST, IMGSZ, bs=4), f"{WORK}/subs/{de_best}.csv")

## 10. Final pipeline

In [ ]:
start = yo.mAP.astype(float).idxmax()
SC = YOLO_RUNS[start]
model = YOLO(f"{WORK}/runs/{start}/weights/best.pt")
print("start from", start)

VAL_LOW = []
for key, p, s in VAL:
    im = VAL_IMG[key]
    f = 680 / max(im["width"], im["height"])
    dst = p.replace("/val/", "/val680/")
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        Image.open(p).resize((round(im["width"] * f), round(im["height"] * f)), Image.BILINEAR).save(dst)
    VAL_LOW.append((key, dst, f))

def tiles_predict(m, items, imgsz, iou=0.6):
    preds = []
    for key, p, s in items:
        img = np.array(Image.open(p).convert("RGB"))[:, :, ::-1]
        H, W = img.shape[:2]
        tw, th = int(W / 1.8), int(H / 1.8)
        crops = [(img[y:y + th, x:x + tw], x, y) for y in [0, H - th] for x in [0, W - tw]] + [(img, 0, 0)]
        B, S_, C = [], [], []
        for (crop, x0, y0), r in zip(crops, m.predict([np.ascontiguousarray(c[0]) for c in crops], imgsz=imgsz,
                                                      conf=0.001, iou=iou, max_det=500, verbose=False)):
            B.append(r.boxes.xyxy.cpu() + torch.tensor([x0, y0, x0, y0]))
            S_.append(r.boxes.conf.cpu()); C.append(r.boxes.cls.cpu())
        B, S_, C = torch.cat(B), torch.cat(S_), torch.cat(C)
        keep = batched_nms(B, S_, C.long(), iou)[:500]
        for (x1, y1, x2, y2), sc, c in zip((B[keep] / s).tolist(), S_[keep].tolist(), C[keep].tolist()):
            preds.append(dict(image_id=key, category_id=int(c) + 1, score=round(sc, 5),
                              bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)]))
    return preds

FINAL = {}
def final_eval(name, fn):
    m = evaluate(fn(VAL))
    m["mAP_680"] = evaluate(fn(VAL_LOW))["mAP"]
    m["score"] = round(0.84 * m["mAP"] + 0.16 * m["mAP_680"], 4)
    FINAL[name] = m
    return pd.DataFrame([m], index=[name])[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

### Inference strategies (same weights)

In [ ]:
INFER = {
    "plain":    lambda m, it: ul_predict(m, it, SC["imgsz"]),
    "nms0.5":   lambda m, it: ul_predict(m, it, SC["imgsz"], iou=0.5),
    "nms0.7":   lambda m, it: ul_predict(m, it, SC["imgsz"], iou=0.7),
    "size+320": lambda m, it: ul_predict(m, it, SC["imgsz"] + 320),
    "tta":      lambda m, it: ul_predict(m, it, SC["imgsz"], augment=True),
    "tiles":    lambda m, it: tiles_predict(m, it, SC["imgsz"]),
}
for k, fn in INFER.items():
    final_eval("A_" + k, lambda it, fn=fn: fn(model, it))
inf = pd.DataFrame(FINAL).T
best_inf = inf.score.astype(float).idxmax()[2:]
print("best inference:", best_inf)
inf

### Data strategy: grey out ignored regions in train frames

In [ ]:
D_MASK = prepare(f"{DATA}/mask", mask_ignore=True)
m_mask = ul_train("final_mask", SC["src"], SC["imgsz"], EPOCHS_YOLO, SC["batch"], load=SC.get("load"),
                  data=f"{D_MASK}/data.yaml", optimizer="SGD", lr0=0.01, close_mosaic=3, max_det=500, cache=False)
final_eval("B_mask", lambda it: ul_predict(m_mask, it, SC["imgsz"]))

### Model change: YOLOv8m

In [ ]:
use_mask = FINAL["B_mask"]["score"] > FINAL["A_plain"]["score"]
src_m = SC["src"].replace("yolov8s", "yolov8m")
m_big = ul_train("final_yolov8m", src_m, SC["imgsz"], EPOCHS_YOLO, SC["batch"],
                 load="yolov8m.pt" if SC.get("load") else None,
                 data=f"{D_MASK if use_mask else D0}/data.yaml", optimizer="SGD", lr0=0.01, close_mosaic=3,
                 max_det=500, cache=False)
final_eval("C_yolov8m", lambda it: ul_predict(m_big, it, SC["imgsz"]))
pd.DataFrame(FINAL).T[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

### Final submission

In [ ]:
cands = {"A_plain": model, "B_mask": m_mask, "C_yolov8m": m_big}
final_name = max(cands, key=lambda k: FINAL[k]["score"])
print("final:", final_name, "+", best_inf)
final_eval("final", lambda it: INFER[best_inf](cands[final_name], it))
write_sub(INFER[best_inf](cands[final_name], TEST), f"{WORK}/submission.csv")
pd.DataFrame(FINAL).T[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

## 11. Summary

In [ ]:
allm = pd.DataFrame(RESULTS).T
display(allm[["family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms", "FPS"]])
best_fam = allm.loc[allm.groupby("family").mAP.apply(lambda s: s.astype(float).idxmax())]
size_plot(best_fam.set_index("family"), "best_per_family_by_size")
pd.DataFrame(FINAL).T.to_csv(f"{WORK}/final_results.csv")
print(os.listdir(f"{WORK}/subs"))